# 05 — Research Synthesis: Map-Reduce & Refine

**Track:** Intermediate · **Stage:** Generation & Synthesis

When a user asks a complex question (e.g. "Summarize all the operational risks of migrating to the new database"), the answer might span 20 or 50 documents. You cannot fit 50 documents into a standard context window. 

In this deep dive, you will build **Map-Reduce** and **Refine** chains using LangChain. We will also introduce conflicting mock data to see how synthesis handles disagreement across sources.

## Setup: LangChain LCEL (LangChain Expression Language)

We will use modern LCEL to construct our synthesis chains.

In [ ]:
# !pip install langchain langchain-core

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms.fake import FakeListLLM
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def print_doc(doc, prefix=""):
    print(f"{prefix}[{doc.metadata['source']}] {doc.page_content}")

## 1. The Conflicting Corpus

We have 4 documents discussing the migration to "Atlas". Notice the disagreement between `eng_report.md` and `qa_testing.md` regarding latency.

In [ ]:
docs = [
    Document(
        page_content="The Atlas migration will cut our AWS bill by 30% due to better resource pooling.",
        metadata={"source": "finance_memo.md"}
    ),
    Document(
        page_content="Atlas brings our P99 latency down to 45ms. We are ready for production.",
        metadata={"source": "eng_report.md"}
    ),
    Document(
        page_content="During stress testing, Atlas P99 latency spiked to 800ms under heavy load. Do not deploy yet.",
        metadata={"source": "qa_testing.md"}
    ),
    Document(
        page_content="Security sign-off for Atlas is complete. No critical vulnerabilities found.",
        metadata={"source": "security_audit.md"}
    )
]

## 2. The Map-Reduce Pattern

**Map:** Run an LLM over *each* document individually to extract claims relevant to the question.
**Reduce:** Run a final LLM over the combined claims to synthesize a final answer.

In [ ]:
question = "What are the risks and benefits of the Atlas migration?"

# ---------------- MAP PHASE ----------------
map_template = """
Extract claims related to the following question from the document. 
If the document is irrelevant, output 'NONE'.
Question: {question}
Document: {context}
"""
map_prompt = ChatPromptTemplate.from_template(map_template)

# Simulating the Map LLM output for the 4 documents
map_responses = [
    "Benefit: Cuts AWS bill by 30%.",
    "Benefit: P99 latency down to 45ms.",
    "Risk: P99 latency spikes to 800ms under load.",
    "Benefit: Security sign-off complete."
]
map_llm = FakeListLLM(responses=map_responses)
map_chain = map_prompt | map_llm | StrOutputParser()

# We mock the parallel execution here for demonstration
print("--- MAP PHASE OUTPUT ---")
mapped_claims = []
for idx, doc in enumerate(docs):
    claim = map_chain.invoke({"question": question, "context": doc.page_content})
    mapped_claims.append(f"[From {doc.metadata['source']}]: {claim}")
    print(mapped_claims[-1])


# ---------------- REDUCE PHASE ----------------
reduce_template = """
Synthesize the following claims into a final report answering the question.
Highlight any disagreements between sources.

Question: {question}
Claims:
{claims}
"""
reduce_prompt = ChatPromptTemplate.from_template(reduce_template)

# Simulating the Reduce LLM output
final_report = (
    "The Atlas migration offers significant benefits, including a 30% reduction in AWS costs [finance_memo.md] "
    "and complete security sign-off [security_audit.md]. However, there is a major disagreement regarding performance risks. "
    "While Engineering reports P99 latency is down to 45ms [eng_report.md], QA warns of 800ms spikes under heavy load [qa_testing.md]."
)
reduce_llm = FakeListLLM(responses=[final_report])
reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

print("\n--- REDUCE PHASE OUTPUT (FINAL SYNTHESIS) ---")
print(reduce_chain.invoke({"question": question, "claims": "\n".join(mapped_claims)}))

## 3. The Refine Pattern

Instead of mapping in parallel, **Refine** works sequentially. 
- LLM reads Doc 1 -> Drafts an answer.
- LLM reads Doc 2 + Draft Answer -> Updates the answer.
- LLM reads Doc 3 + Draft Answer -> Updates the answer.

**Pros:** Better narrative flow. 
**Cons:** Extremely slow (sequential LLM calls), susceptible to losing earlier context ("recency bias").

In [ ]:
refine_template = """
Your job is to produce a final answer to the question.
We have provided an existing answer up to a certain point: {existing_answer}
We have the opportunity to refine the existing answer (only if needed) with some more context below.
------------
{context}
------------
Given the new context, refine the original answer to better answer the question: {question}
"""
# LangChain provides a built-in load_summarize_chain(chain_type="refine") 
# which handles this loop automatically under the hood.
print("Refine involves a sequential loop over the documents, updating a running state.")

## Reflection

1. **Cost:** Map-Reduce over 50 documents requires 51 LLM calls. If each call is 1 second, it takes 1 second (Map runs in parallel) + 1 second (Reduce) = 2 seconds total latency, but costs 51x the tokens. Refine takes 50 seconds.
2. **Conflict Resolution:** Did the Reduce step arbitrarily pick a "winner" between Engineering and QA? The prompt explicitly instructed it to "Highlight any disagreements". In synthesis, exposing the conflict is always better than hallucinating a consensus.